# Text Preprocessing for NLP

## What is Text Preprocessing?
Text preprocessing is the process of transforming raw text into a clean, structured format suitable for machine learning models. Raw text is noisy, inconsistent, and full of irrelevant information. Preprocessing standardizes it.

## Why is it Important?
- Reduces vocabulary size (memory & computation)
- Removes noise that confuses models
- Normalizes text for consistent representation
- Converts unstructured text into structured features

---

## 1. Tokenization

Tokenization splits text into smaller units (tokens). Three main levels:

### 1.1 Word Tokenization
Splits text into words: `"Hello world"` → `["Hello", "world"]`

### 1.2 Sentence Tokenization
Splits text into sentences using punctuation and context.

### 1.3 Subword Tokenization
Breaks words into subword units critical for handling unknown words:
- **BPE (Byte Pair Encoding)**: Merges most frequent byte pairs iteratively
- **WordPiece**: Used by BERT, maximizes likelihood of training data
- **SentencePiece**: Language-agnostic, treats text as raw unicode
- **Unigram**: Probabilistic subword segmentation

---

## 2. Text Normalization

### 2.1 Lowercasing
Converts all text to lowercase: `"Apple"` → `"apple"`

### 2.2 Punctuation Removal
Removes punctuation marks (context-dependent sometimes useful).

### 2.3 Stop Words
Common words with little semantic meaning: *the, is, at, which, on*

### 2.4 Contractions
Expand contractions: `"don't"` → `"do not"`

### 2.5 Unicode & HTML Cleaning
Remove HTML tags, normalize unicode characters, handle emojis.

---

## 3. Stemming

Stemming reduces words to their root/base form by removing suffixes (crude, rule-based):
- `"running"` → `"run"`, `"studies"` → `"studi"`

### Algorithms:
- **Porter Stemmer**: Most common, 5-phase suffix stripping rules
- **Snowball (Porter2)**: Improved Porter, supports multiple languages
- **Lancaster**: Aggressive stemmer, faster but over-stems

---

## 4. Lemmatization

Lemmatization returns the dictionary base form (lemma) using vocabulary and morphological analysis:
- `"better"` → `"good"`, `"running"` → `"run"`

More accurate than stemming but slower. Uses WordNet lexical database.

---

## 5. N-grams

N-grams are contiguous sequences of N tokens:
- **Unigrams** (N=1): `["I", "love", "NLP"]`
- **Bigrams** (N=2): `["I love", "love NLP"]`
- **Trigrams** (N=3): `["I love NLP"]`

Capture local word order and phrases.

---

## 6. Bag of Words (BoW)

Represents text as a vector of word counts, ignoring order:

$$\text{BoW}(d) = [\text{count}(w_1, d), \text{count}(w_2, d), ..., \text{count}(w_V, d)]$$

- Simple and effective baseline
- Loses word order and context
- High dimensionality (vocabulary size)

---

## 7. TF-IDF (Term Frequency - Inverse Document Frequency)

Weights words by how important they are to a document relative to the corpus.

### Term Frequency:
$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

### Inverse Document Frequency:
$$\text{IDF}(t) = \log\frac{N}{df(t) + 1}$$

where $N$ = total documents, $df(t)$ = documents containing term $t$

### TF-IDF Score:
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{df(t)}$$

High TF-IDF means the word is frequent in this document but rare across all documents → distinctive word.

---

## 8. BM25 (Best Match 25)

BM25 is a ranking function that improves on TF-IDF with term frequency saturation and document length normalization:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Where:
- $f(q_i, D)$ = frequency of query term $q_i$ in document $D$
- $|D|$ = document length
- $\text{avgdl}$ = average document length in corpus
- $k_1 \in [1.2, 2.0]$ = term frequency saturation parameter
- $b = 0.75$ = length normalization parameter

BM25 is the gold standard for traditional information retrieval and keyword search.

---

## 9. spaCy Pipeline

spaCy processes text through a pipeline: `tokenizer → tagger → parser → NER → ...`

Each component adds annotations to the Doc object.

In [1]:
# Install required libraries
# !pip install nltk spacy rank_bm25 scikit-learn
# !python -m spacy download en_core_web_sm

import nltk
import re
import string
from collections import Counter

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

sample_text = """
Natural Language Processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and 
human language. It's amazing how computers are learning to understand us!
The applications include machine translation, sentiment analysis, and chatbots.
"""

print("Original text:")
print(sample_text)

Original text:

Natural Language Processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and 
human language. It's amazing how computers are learning to understand us!
The applications include machine translation, sentiment analysis, and chatbots.



In [2]:
# ============================================================
# TOKENIZATION
# ============================================================
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer
from nltk.tokenize import RegexpTokenizer

# Word tokenization
word_tokens = word_tokenize(sample_text)
print("Word tokens (first 15):", word_tokens[:15])

# Sentence tokenization
sent_tokens = sent_tokenize(sample_text)
print("\nSentence tokens:")
for i, sent in enumerate(sent_tokens):
    print(f"  [{i+1}] {sent.strip()}")

# Regex tokenizer (only alphabetic tokens)
regex_tokenizer = RegexpTokenizer(r'\w+')
regex_tokens = regex_tokenizer.tokenize(sample_text)
print("\nRegex tokens (first 15):", regex_tokens[:15])

Word tokens (first 15): ['Natural', 'Language', 'Processing', '(', 'NLP', ')', 'is', 'a', 'subfield', 'of', 'linguistics', ',', 'computer', 'science', ',']

Sentence tokens:
  [1] Natural Language Processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and 
human language.
  [2] It's amazing how computers are learning to understand us!
  [3] The applications include machine translation, sentiment analysis, and chatbots.

Regex tokens (first 15): ['Natural', 'Language', 'Processing', 'NLP', 'is', 'a', 'subfield', 'of', 'linguistics', 'computer', 'science', 'and', 'artificial', 'intelligence', 'concerned']


In [3]:
# ============================================================
# TEXT NORMALIZATION
# ============================================================
from nltk.corpus import stopwords

def clean_text(text):
    """Full text cleaning pipeline"""
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Expand contractions (simple examples)
    contractions = {"it's": "it is", "don't": "do not", "can't": "cannot",
                    "won't": "will not", "i'm": "i am", "it's": "it is"}
    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned = clean_text(sample_text)
print("Cleaned text:", cleaned)

# Remove stop words
stop_words = set(stopwords.words('english'))
tokens = word_tokenize(cleaned)
filtered = [w for w in tokens if w not in stop_words]
print("\nAfter stop word removal:", filtered)

Cleaned text: natural language processing nlp is a subfield of linguistics computer science and artificial intelligence concerned with the interactions between computers and human language it is amazing how computers are learning to understand us the applications include machine translation sentiment analysis and chatbots

After stop word removal: ['natural', 'language', 'processing', 'nlp', 'subfield', 'linguistics', 'computer', 'science', 'artificial', 'intelligence', 'concerned', 'interactions', 'computers', 'human', 'language', 'amazing', 'computers', 'learning', 'understand', 'us', 'applications', 'include', 'machine', 'translation', 'sentiment', 'analysis', 'chatbots']


In [4]:
# ============================================================
# STEMMING - Three algorithms compared
# ============================================================
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter = PorterStemmer()
snowball = SnowballStemmer('english')
lancaster = LancasterStemmer()

test_words = ['running', 'studies', 'generously', 'happiness', 'computational', 
              'processing', 'languages', 'learning', 'algorithms']

print(f"{'Word':<20} {'Porter':<15} {'Snowball':<15} {'Lancaster':<15}")
print("-" * 65)
for word in test_words:
    print(f"{word:<20} {porter.stem(word):<15} {snowball.stem(word):<15} {lancaster.stem(word):<15}")

Word                 Porter          Snowball        Lancaster      
-----------------------------------------------------------------
running              run             run             run            
studies              studi           studi           study          
generously           gener           generous        gen            
happiness            happi           happi           happy          
computational        comput          comput          comput         
processing           process         process         process        
languages            languag         languag         langu          
learning             learn           learn           learn          
algorithms           algorithm       algorithm       algorithm      


In [5]:
# ============================================================
# LEMMATIZATION
# ============================================================
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

lemmatizer = WordNetLemmatizer()

# Lemmatize with POS tag for accuracy
def get_wordnet_pos(tag):
    from nltk.corpus import wordnet
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('N'): return wordnet.NOUN
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

words_pos = pos_tag(test_words)
print(f"{'Word':<20} {'Stem (Porter)':<20} {'Lemma':<20}")
print("-" * 60)
for word, pos in words_pos:
    wn_pos = get_wordnet_pos(pos)
    lemma = lemmatizer.lemmatize(word, pos=wn_pos)
    stem = porter.stem(word)
    print(f"{word:<20} {stem:<20} {lemma:<20}")

Word                 Stem (Porter)        Lemma               
------------------------------------------------------------


running              run                  run                 
studies              studi                study               
generously           gener                generously          
happiness            happi                happiness           
computational        comput               computational       
processing           process              processing          
languages            languag              language            
learning             learn                learn               
algorithms           algorithm            algorithm           


In [6]:
# ============================================================
# N-GRAMS
# ============================================================
from nltk.util import ngrams

tokens = word_tokenize("I love natural language processing with Python")

print("Unigrams:", list(ngrams(tokens, 1)))
print("Bigrams: ", list(ngrams(tokens, 2)))
print("Trigrams:", list(ngrams(tokens, 3)))

# Most common bigrams in a corpus
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

finder = BigramCollocationFinder.from_words(word_tokenize(sample_text.lower()))
finder.apply_freq_filter(2)  # At least 2 occurrences
print("\nTop bigrams by PMI:", finder.nbest(BigramAssocMeasures.pmi, 5))

Unigrams: [('I',), ('love',), ('natural',), ('language',), ('processing',), ('with',), ('Python',)]
Bigrams:  [('I', 'love'), ('love', 'natural'), ('natural', 'language'), ('language', 'processing'), ('processing', 'with'), ('with', 'Python')]
Trigrams: [('I', 'love', 'natural'), ('love', 'natural', 'language'), ('natural', 'language', 'processing'), ('language', 'processing', 'with'), ('processing', 'with', 'Python')]

Top bigrams by PMI: [(',', 'and')]


In [7]:
# ============================================================
# BAG OF WORDS & TF-IDF
# ============================================================
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
import numpy as np

corpus = [
    "Natural language processing is amazing",
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks",
    "Natural language understanding requires deep learning",
    "Artificial intelligence and machine learning are transforming technology"
]

# Bag of Words
bow_vectorizer = CountVectorizer(stop_words='english', max_features=10)
bow_matrix = bow_vectorizer.fit_transform(corpus)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out())
print("Bag of Words matrix:")
print(bow_df)

# TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=10)
tfidf_matrix = tfidf.fit_transform(corpus)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())
print("\nTF-IDF matrix:")
print(tfidf_df.round(3))

Bag of Words matrix:
   amazing  artificial  deep  intelligence  language  learning  machine  \
0        1           0     0             0         1         0        0   
1        0           1     0             1         0         1        1   
2        0           0     1             0         0         1        0   
3        0           0     1             0         1         1        0   
4        0           1     0             1         0         1        1   

   natural  networks  neural  
0        1         0       0  
1        0         0       0  
2        0         1       1  
3        1         0       0  
4        0         0       0  

TF-IDF matrix:
   amazing  artificial   deep  intelligence  language  learning  machine  \
0    0.659       0.000  0.000         0.000     0.532     0.000    0.000   
1    0.000       0.535  0.000         0.535     0.000     0.374    0.535   
2    0.000       0.000  0.468         0.000     0.000     0.327    0.000   
3    0.000       0.000

In [8]:
# ============================================================
# TF-IDF FROM SCRATCH
# ============================================================
import math

def compute_tf(text):
    """Compute term frequency for a document"""
    words = text.lower().split()
    tf = Counter(words)
    total = len(words)
    return {word: count/total for word, count in tf.items()}

def compute_idf(corpus):
    """Compute inverse document frequency for corpus"""
    N = len(corpus)
    all_words = set(word for doc in corpus for word in doc.lower().split())
    idf = {}
    for word in all_words:
        df = sum(1 for doc in corpus if word in doc.lower().split())
        idf[word] = math.log(N / (df + 1))
    return idf

def compute_tfidf(doc, corpus):
    tf = compute_tf(doc)
    idf = compute_idf(corpus)
    return {word: tf_val * idf.get(word, 0) for word, tf_val in tf.items()}

tfidf_scores = compute_tfidf(corpus[0], corpus)
sorted_scores = sorted(tfidf_scores.items(), key=lambda x: x[1], reverse=True)
print("TF-IDF scores for first document (from scratch):")
for word, score in sorted_scores[:5]:
    print(f"  '{word}': {score:.4f}")

TF-IDF scores for first document (from scratch):
  'processing': 0.1833
  'amazing': 0.1833
  'natural': 0.1022
  'language': 0.1022
  'is': 0.1022


In [9]:
# ============================================================
# BM25
# ============================================================
from rank_bm25 import BM25Okapi

# Tokenize corpus
tokenized_corpus = [doc.lower().split() for doc in corpus]

# BM25 with default k1=1.5, b=0.75
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

query = "deep learning neural networks"
tokenized_query = query.lower().split()
scores = bm25.get_scores(tokenized_query)

print(f"BM25 scores for query: '{query}'")
for i, (doc, score) in enumerate(zip(corpus, scores)):
    print(f"  Doc {i+1} (score={score:.4f}): {doc[:50]}...")

# Top N results
top_n = bm25.get_top_n(tokenized_query, corpus, n=2)
print("\nTop 2 results:")
for doc in top_n:
    print(f"  → {doc}")

BM25 scores for query: 'deep learning neural networks'
  Doc 1 (score=0.0000): Natural language processing is amazing...
  Doc 2 (score=0.1699): Machine learning is a subset of artificial intelli...
  Doc 3 (score=3.0200): Deep learning uses neural networks...
  Doc 4 (score=0.5407): Natural language understanding requires deep learn...
  Doc 5 (score=0.1699): Artificial intelligence and machine learning are t...

Top 2 results:
  → Deep learning uses neural networks
  → Natural language understanding requires deep learning


In [10]:
# ============================================================
# SPACY PIPELINE
# ============================================================
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    
    text = "Apple is looking at buying U.K. startup for $1 billion in London."
    doc = nlp(text)
    
    print("spaCy Pipeline components:", nlp.pipe_names)
    print()
    
    # Tokens
    print("Tokens:")
    print(f"{'Token':<15} {'Lemma':<15} {'POS':<10} {'Tag':<10} {'Dep':<15} {'StopWord':<10}")
    print("-" * 75)
    for token in doc:
        print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<10} {token.tag_:<10} {token.dep_:<15} {str(token.is_stop):<10}")
    
    # Named Entities
    print("\nNamed Entities:")
    for ent in doc.ents:
        print(f"  {ent.text:20} → {ent.label_:10} ({spacy.explain(ent.label_)})")
    
    # Noun chunks
    print("\nNoun Chunks:")
    for chunk in doc.noun_chunks:
        print(f"  '{chunk.text}' root: '{chunk.root.text}'")

except OSError:
    print("spaCy model not found. Run: python -m spacy download en_core_web_sm")

spaCy Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

Tokens:
Token           Lemma           POS        Tag        Dep             StopWord  
---------------------------------------------------------------------------
Apple           Apple           PROPN      NNP        nsubj           False     
is              be              AUX        VBZ        aux             True      
looking         look            VERB       VBG        ROOT            False     
at              at              ADP        IN         prep            True      
buying          buy             VERB       VBG        pcomp           False     
U.K.            U.K.            PROPN      NNP        nsubj           False     
startup         startup         VERB       VBD        ccomp           False     
for             for             ADP        IN         prep            True      
$               $               SYM        $          quantmod        False     
1     

In [11]:
# ============================================================
# SUBWORD TOKENIZATION with Hugging Face
# ============================================================
try:
    from transformers import AutoTokenizer
    
    # BERT uses WordPiece
    bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    
    test = "Natural language processing is transformational!"
    bert_tokens = bert_tokenizer.tokenize(test)
    bert_ids = bert_tokenizer.encode(test)
    
    print("BERT (WordPiece) tokens:", bert_tokens)
    print("Token IDs:", bert_ids)
    print("Vocabulary size:", bert_tokenizer.vocab_size)
    
    # GPT-2 uses BPE
    gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')
    gpt2_tokens = gpt2_tokenizer.tokenize(test)
    print("\nGPT-2 (BPE) tokens:", gpt2_tokens)
    print("GPT-2 vocab size:", gpt2_tokenizer.vocab_size)
    
except Exception as e:
    print(f"Transformers not available: {e}")

BERT (WordPiece) tokens: ['natural', 'language', 'processing', 'is', 'transformation', '##al', '!']
Token IDs: [101, 3019, 2653, 6364, 2003, 8651, 2389, 999, 102]
Vocabulary size: 30522



GPT-2 (BPE) tokens: ['Natural', 'Ġlanguage', 'Ġprocessing', 'Ġis', 'Ġtransform', 'ational', '!']
GPT-2 vocab size: 50257


## Additional Learning Resources

### Documentation
- [spaCy Documentation](https://spacy.io/usage) Industrial-strength NLP library
- [NLTK Book](https://www.nltk.org/book/) Free online textbook for NLP with NLTK
- [Hugging Face Tokenizers](https://huggingface.co/docs/tokenizers/) Fast tokenizers library
- [scikit-learn Text Feature Extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)

### Papers
- [BPE Paper: Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909) Sennrich et al., 2016
- [BM25: The Probabilistic Relevance Framework](https://www.staff.city.ac.uk/~seb/papers/robertson_sparck_jones_1976.pdf) Robertson & Sparck Jones
- [SentencePiece](https://arxiv.org/abs/1808.06226) Language-agnostic tokenization

### Courses & Tutorials
- [Stanford CS224n: NLP with Deep Learning](https://web.stanford.edu/class/cs224n/) Full course with slides and videos
- [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course) Free, comprehensive
- [fast.ai NLP Course](https://www.fast.ai/) Practical approach
- [Speech and Language Processing Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/) Free textbook

### Books
- *Natural Language Processing with Python* Bird, Klein & Loper (NLTK Book)
- *Speech and Language Processing* Jurafsky & Martin (3rd ed., free online)
- *Natural Language Processing with Transformers* Tunstall et al.